# 🚗 End-to-End YOLOv8 Data Pipeline: Autonomous Driving
### **Preparing Self-Driving Car Data for YOLOv8**

This notebook serves as a **complete data pipeline** designed to transform raw data into a format fully ready for training a **YOLOv8** model, addressing issues like class imbalance and directory structure.

---

### 🛠️ **What does this notebook do? (Workflow)**

1.  **📥 Data Ingestion:**
    * Reads data from the `export` folder (where all images and labels are currently located).
    * Automatically detects class names from the original `data.yaml` file.

2.  **🔀 Shuffling & Splitting:**
    * Randomly shuffles images to ensure a fair distribution.
    * Automatically splits the data into 3 sets using standard ratios:
        * **Train:** 70% (for training).
        * **Valid:** 20% (for validation during training).
        * **Test:** 10% (for final testing).

3.  **🖼️ Preprocessing:**
    * **Resize:** Standardizes all image sizes to **416x416** to ensure faster training.
    * **Label Formatting:** Moves label files (`.txt`) to match images in the new folders.

4.  **✨ Data Augmentation:**
    * Applies **Albumentations** techniques to the training data (**Train set only**) to increase model accuracy and prevent overfitting.
    * Techniques used: `Rotation`, `Flip`, `Brightness`, `Blur`, `GaussNoise`.

5.  **⚙️ Configuration & Export:**
    * Generates a new `data.yaml` file containing the correct paths (`/kaggle/working/...`).
    * Compresses the final directory into a **`final_dataset.zip`** file, ready for download or immediate use.

---

### 📊 **Classes Included:**
This project covers the detection of vital objects for autonomous driving, including:
* `Car`, `Truck`, `Biker`, `Pedestrian`
* 🚦 **Traffic Lights:** (Green, Red, Yellow, Left) - *Essential for autonomous decision making.*

---
**🚀 Ready to Run? Just execute all cells below!**

# 0. Import Libraries
Import necessary libraries for file handling, image processing (OpenCV), and augmentation (Albumentations).

In [ ]:
!pip install roboflow
import os
import glob
import shutil
import cv2
import yaml
import random
import math
import numpy as np
import albumentations as A
from tqdm.notebook import tqdm
from roboflow import Roboflow
import matplotlib.pyplot as plt

In [ ]:
# Download Dataset
rf = Roboflow(api_key="566r25x43JupkeFgvfe7")
project = rf.workspace("test-smutr").project("my-first-project-uh0wb")
version = project.version(2)
dataset = version.download("yolov8")

# 1. Define Paths
Set up the paths for the source dataset (flat structure) and the destination directory.

In [ ]:
SOURCE_DIR = "/kaggle/working/My-First-Project-2"
SOURCE_YAML = "/kaggle/working/My-First-Project-2/data.yaml"
FINAL_DIR = "/kaggle/working/Ready_Dataset"

# 1.1. Define Configuration 
Set up the Configuration Parameters.

In [ ]:
# Parameters
TARGET_SIZE = 416
# How many times to augment rare classes
MAX_AUG_FACTOR = 3 

# 2. Create Directory Structure
Create `images` and `labels` folders for Train, Validation, and Test sets.

In [ ]:
subsets = ['train', 'valid', 'test']

for subset in subsets:
    os.makedirs(os.path.join(FINAL_DIR, subset, 'images'), exist_ok=True)
    os.makedirs(os.path.join(FINAL_DIR, subset, 'labels'), exist_ok=True)

# 4. Define Processing Function
A function to resize images to 416x416 and move corresponding labels to the target split folder.

In [ ]:
def Preprocess(subset):
    # Define source paths
    src_img_dir = os.path.join(SOURCE_DIR, subset, "images")
    src_lbl_dir = os.path.join(SOURCE_DIR, subset, "labels")
    
    # Define destination paths
    dest_img_dir = os.path.join(FINAL_DIR, subset, "images")
    dest_lbl_dir = os.path.join(FINAL_DIR, subset, "labels")
    
    # Create directories
    os.makedirs(dest_img_dir, exist_ok=True)
    os.makedirs(dest_lbl_dir, exist_ok=True)
    
    # Check if source directory exists
    if not os.path.exists(src_img_dir):
        print(f"⚠️ Warning: {subset}/images not found in source.")
        return

    # Get all images
    images = glob.glob(os.path.join(src_img_dir, "*.*"))
    # Filter for valid image extensions
    images = [f for f in images if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    print(f"🔄 Processing {subset}: {len(images)} images...")

    for img_path in tqdm(images, desc=f"Resizing {subset}"):
        filename = os.path.basename(img_path)
        base_name = os.path.splitext(filename)[0]
        
        # 1. Resize Image (No try-except)
        img = cv2.imread(img_path)
        if img is None: continue
        
        img_resized = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE))
        cv2.imwrite(os.path.join(dest_img_dir, filename), img_resized)
        
        # 2. Process Label (Transfer & Clamp)
        src_lbl = os.path.join(src_lbl_dir, base_name + ".txt")
        dst_lbl = os.path.join(dest_lbl_dir, base_name + ".txt")
        
        if os.path.exists(src_lbl):
            with open(src_lbl, 'r') as f_in, open(dst_lbl, 'w') as f_out:
                for line in f_in:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls = parts[0]
                        coords = [float(x) for x in parts[1:5]]
                        
                        # Essential step: Ensure values are between 0 and 1
                        coords = [max(0.0, min(1.0, x)) for x in coords]
                        
                        f_out.write(f"{cls} {' '.join(f'{x:.6f}' for x in coords)}\n")
        else:
            # Create empty file for background images
            open(dst_lbl, 'w').close()


# 5. Execute Process
Divide the shuffled data into Train, Validation, and Test sets based on ratios.

In [ ]:
# Run processing for all subsets
Preprocess('train')
Preprocess('valid')
Preprocess('test')
print("✅ Resize & Preparation Complete!")

# 6. Define Augmentation Pipeline
Set up transformations (Flip, Brightness, Noise) for the training set.

In [ ]:
aug_pipeline = A.Compose([
    A.Affine(translate_percent=0.1, scale=(0.9, 1.1), rotate=(-10, 10), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Blur(blur_limit=3, p=0.2),
    A.GaussNoise(p=0.2)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

# 6.1. Calculate Class Distribution & Factors
Scan all label files to count classes,Rare classes get higher factors, up to MAX_AUG_FACTOR

In [ ]:
# 1. Calculate Class Distribution & Factors
train_lbl_dir = os.path.join(FINAL_DIR, "train", "labels")
counts = {}

# Scan files
for f in glob.glob(os.path.join(train_lbl_dir, "*.txt")):
    if os.path.getsize(f) > 0:
        with open(f, 'r') as file:
            for line in file:
                parts = line.split()
                if len(parts) > 0:
                    cls = int(float(parts[0]))
                    counts[cls] = counts.get(cls, 0) + 1

# Calculate Factors
if counts:
    max_count = max(counts.values())
    factors = {k: min(MAX_AUG_FACTOR, math.ceil(max_count / v)) for k, v in counts.items()}
    print("Class Factors:", factors)
else:
    factors = {}

# 7. Define Augmentation Loop
Iterate through the training set and generate 1 augmented copy for each image.

In [ ]:
# 2. Define & Run Augmentation
def run_augmentation():
    img_dir = os.path.join(FINAL_DIR, "train", "images")
    lbl_dir = os.path.join(FINAL_DIR, "train", "labels")
    
    files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    
    for filename in tqdm(files, desc="Augmenting"):
        if "_aug_" in filename: continue
        
        base_name = os.path.splitext(filename)[0]
        img_path = os.path.join(img_dir, filename)
        lbl_path = os.path.join(lbl_dir, base_name + ".txt")
        
        img = cv2.imread(img_path)
        if img is None: continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        bboxes = []
        classes = []
        
        if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 5:
                        c = int(float(parts[0]))
                        # Read Raw Coordinates
                        xc, yc, w, h = [float(x) for x in parts[1:5]]
                        
                        # --- GEOMETRY FIX: Recalculate valid box inside [0,1] ---
                        # 1. Convert to Corners (x1, y1, x2, y2)
                        x1 = xc - w / 2
                        y1 = yc - h / 2
                        x2 = xc + w / 2
                        y2 = yc + h / 2
                        
                        # 2. Clip Corners to Image Boundaries
                        x1 = max(0.0, min(1.0, x1))
                        y1 = max(0.0, min(1.0, y1))
                        x2 = max(0.0, min(1.0, x2))
                        y2 = max(0.0, min(1.0, y2))
                        
                        # 3. Convert back to YOLO (Center, Width, Height)
                        new_xc = (x1 + x2) / 2
                        new_yc = (y1 + y2) / 2
                        new_w = x2 - x1
                        new_h = y2 - y1
                        
                        # 4. Validate Box Size (Remove tiny/empty boxes)
                        if new_w <= 0.001 or new_h <= 0.001: continue
                        
                        classes.append(c)
                        bboxes.append([new_xc, new_yc, new_w, new_h])
        
        if not classes: continue
        
        # Augmentation Loop
        current_factor = max([factors.get(c, 1) for c in classes])
        
        for i in range(current_factor - 1):
            # Apply Pipeline
            augmented = aug_pipeline(image=img_rgb, bboxes=bboxes, class_labels=classes)
            
            if augmented['bboxes']:
                aug_name = f"{base_name}_aug_{i}"
                
                # Save Image
                cv2.imwrite(os.path.join(img_dir, aug_name + ".jpg"), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
                
                # Save Label
                with open(os.path.join(lbl_dir, aug_name + ".txt"), 'w') as f:
                    for c, b in zip(augmented['class_labels'], augmented['bboxes']):
                        # Final safety clamp for output
                        b = [max(0.0, min(1.0, x)) for x in b]
                        f.write(f"{c} {' '.join(f'{x:.6f}' for x in b)}\n")

# 8. Execute Augmentation
Start the augmentation process.

In [ ]:
run_augmentation()

# 9. Generate Configuration
Create the `data.yaml` file required by YOLOv8.

In [ ]:
# Load original names
with open(SOURCE_YAML, 'r') as f:
    config = yaml.safe_load(f)
    class_names = config.get('names', [])

yaml_data = {
    'path': os.path.abspath(FINAL_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(class_names),
    'names': class_names
}

with open(os.path.join(FINAL_DIR, "data.yaml"), 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print("✅ data.yaml created.")

# 10. Zip Dataset
Compress the prepared dataset for download or use in training.

In [ ]:
folder_path = "/kaggle/working/My-First-Project-2"  
shutil.rmtree(folder_path)

In [ ]:
# output_zip = "/kaggle/working/Ready_Dataset"
# shutil.make_archive(output_zip, 'zip', FINAL_DIR)
# print(f"✅ Dataset zipped: {output_zip}.zip")

In [ ]:
# import os
# from IPython.display import FileLink

# # ده المسار اللي الملف المضغوط اتعمل فيه في الخطوة الأخيرة
# zip_file_path = 'Ready_Dataset.zip' 

# # التأكد إن الملف موجود
# if os.path.exists(zip_file_path):
#     print("✅ الملف جاهز! اضغط على الرابط أدناه للتحميل:")
#     display(FileLink(zip_file_path))
# else:
#     print("❌ الملف مش موجود! اتأكد إنك شغلت خلية الضغط (shutil.make_archive) اللي قبل دي.")